# Batch face-only swap (Krea2 model, hair/clothes/background preserved)

Uses the same Krea2 identity-edit model as the head-swap work, but with a **face-only prompt**: only the facial features (bone structure, eyes, nose, mouth, skin) change. Hair, clothing, pose and background are explicitly instructed to stay exactly as they are, and the route is forced to `crop_stitch` -- only a small face crop is ever regenerated and pasted back onto the untouched original, so nothing outside that crop can drift.

Upload 13 body/face pairs and run all of them in one pass.

**3 cells: Setup -> Upload -> Run.**

## 1 · Setup

In [ ]:
from pathlib import Path
import subprocess, os

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-only"

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU, then Run all."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True
)

os.chdir(REPO)
# ComfyUI + deps + Krea2 Identity Edit nodes/weights (Drive-cached after first run).
subprocess.run(["bash", "scripts/setup_colab.sh", "--krea2"], check=True, cwd=str(REPO))
print("Setup complete.")


## 2 · Upload 13 pairs

Upload all 13 **body/scene** photos first (select all at once), then all 13 **face/identity donor** photos in the *same order*. The cell prints the resulting pairing so you can check it before running anything.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image
import io

UPLOAD_DIR = Path("/content/face_swap_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Upload all BODY/scene photos (select all 13 at once)")
body_uploads = files.upload()
print()
print("Upload all FACE/identity donor photos, in the SAME ORDER as the bodies above (select all 13 at once)")
face_uploads = files.upload()

body_names = sorted(body_uploads.keys())
face_names = sorted(face_uploads.keys())
assert len(body_names) == len(face_names), (
    f"Got {len(body_names)} body images and {len(face_names)} face images -- "
    "these must match 1:1. Re-run this cell with matching sets."
)

pairs = []
print()
print("Pairing (verify this matches what you intended):")
for i, (bn, fn) in enumerate(zip(body_names, face_names), start=1):
    bp = UPLOAD_DIR / f"body_{i:02d}.png"
    fp = UPLOAD_DIR / f"face_{i:02d}.png"
    Image.open(io.BytesIO(body_uploads[bn])).convert("RGB").save(bp)
    Image.open(io.BytesIO(face_uploads[fn])).convert("RGB").save(fp)
    pairs.append((bp, fp))
    print(f"  {i:2d}. body={bn!r}  <->  face={fn!r}")

print(f"\n{len(pairs)} pairs ready.")


## 3 · Run all pairs

Face-only prompt, `crop_stitch` forced (no full-frame regeneration -- nothing outside the face crop is ever touched), `preserve_expression=True` (keeps the target's own expression), `headwear_prompt_policy` and the hair-replacement clause both off (the base head-swap prompt otherwise explicitly instructs the model to replace hair, which is the opposite of what this needs).

In [ ]:
from pathlib import Path
import sys, os, time

REPO = Path("/content/headswap_V2")
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline
from PIL import Image
from IPython.display import display, Markdown

FACE_SWAP_PROMPT = (
    "Replace only the facial features in the first image with the facial "
    "features from the second image -- bone structure, jawline, eyebrows, "
    "eyes, nose, mouth, and skin -- matching the exact facial identity of "
    "the second person. "
    "Do not change the hair, hairstyle, hairline, hair length, or hair "
    "colour in any way -- keep the first person's own hair exactly as it "
    "already is. "
    "CRITICAL: copy the facial expression from the first image exactly -- "
    "if they are smiling, the result must smile the same way; if they are "
    "not smiling, do not add a smile. Keep mouth shape, smile/no-smile, "
    "eye gaze, and micro-expressions from the first image only -- never "
    "from the second image. "
    "CRITICAL: keep the head facing the exact same direction as the first "
    "image -- do not turn, rotate, or angle the head to the left or right, "
    "and do not tilt it. The eyes must look in the exact same direction as "
    "the first image. Head yaw, pitch, and roll must exactly match the "
    "first image. "
    "Keep the body, clothing, hair, pose, head rotation, camera angle, "
    "lighting, and background from the first image exactly as they are. "
    "Use only the facial identity from the second image -- do not copy "
    "hair, clothing, collar, or shoulders from the second image. If other "
    "people are visible, leave them completely unchanged. Photorealistic, "
    "natural skin texture, sharp details, lighting matched to the first "
    "image."
)

base_cfg = load_config(str(REPO / "configs" / "krea2_identity_edit.yaml"))
cfg = dict(base_cfg)
cfg.update({
    "prompt": FACE_SWAP_PROMPT,
    # The base prompt/reinforcement both say "replace the hair completely" --
    # this route needs the opposite, so both are switched off and the
    # instruction lives entirely in FACE_SWAP_PROMPT above instead.
    "multi_hair_replace_prompt": False,
    # Default headwear-preserve text says "This is a HEAD SWAP, not a face
    # swap: the second person's hair MUST be transferred" -- exactly wrong
    # here. Off entirely; headwear is just part of "everything but the face
    # stays the same," no special instruction needed.
    "headwear_prompt_policy": False,
    "preserve_expression": True,
    # Force crop_stitch unconditionally -- no auto-upgrade to full-frame
    # regeneration for full-body photos, which is what made clothing/
    # background preservation fragile throughout the head-swap work this
    # session. crop_stitch only ever touches a small face crop.
    "enable_body_route": False,
    "enable_lighting_route": False,
    "multi_person_edit_mode": "crop_stitch",
})

CACHE_DIR = Path("/content/drive/MyDrive/headswap_V2/models")
OUT_DIR = Path("/content/face_swap_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=CACHE_DIR)

results = []
t0 = time.perf_counter()
for i, (body_path, face_path) in enumerate(pairs, start=1):
    body = Image.open(body_path).convert("RGB")
    face = Image.open(face_path).convert("RGB")
    pair_out = OUT_DIR / f"pair_{i:02d}"
    res = pipe.run(body, face, out_dir=pair_out)
    results.append(res)
    print(f"[{i}/{len(pairs)}] done -> {pair_out}")

print(f"\nAll {len(pairs)} pairs done in {time.perf_counter() - t0:.1f}s total.")

for i, res in enumerate(results, start=1):
    display(Markdown(f"### Pair {i}"))
    display(res.image)
